In [1]:
import subprocess, sys, torch

tv = torch.__version__.split("+")[0]
cu = "cu" + torch.version.cuda.replace(".", "")
url = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"

subprocess.run([sys.executable,"-m","pip","install",
    "pyg_lib","torch_scatter","torch_sparse",
    "torch_cluster","torch_spline_conv","-f",url,"-q"], check=False)
subprocess.run([sys.executable,"-m","pip","install","torch_geometric","-q"])
print("✔ PyG installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 105.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 93.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.9 MB/s eta 0:00:00
✔ PyG installed


In [2]:
import shutil, os

CODE = "/kaggle/input/datasets/jerinanan/pgcn-eeg-code"
for f in os.listdir(CODE):
    if f.endswith(".py"):
        shutil.copy(f"{CODE}/{f}", f"/kaggle/working/{f}")
        print(f"  Copied: {f}")

  Copied: kaggle_notebooks.py
  Copied: preprocess_eeg.py
  Copied: pgcn_model.py
  Copied: preprocess_fusion.py
  Copied: pgcn_statistics_plots.py
  Copied: pgcn_train.py
  Copied: pgcn_core.py


In [3]:
with open("/kaggle/working/pgcn_core.py", "r") as f:
    code = f.read()

code = code.replace(
    "            w=self.cw[targets]",
    "            w=self.cw.to(targets.device)[targets]"
)

with open("/kaggle/working/pgcn_core.py", "w") as f:
    f.write(code)

print("✔ Fixed")

✔ Fixed


In [4]:
import os, json

PRE = "/kaggle/input/datasets/jerinanan/pgcn-deap-preprocessed"
subjects = sorted([d for d in os.listdir(PRE) if d.startswith("s")])
print(f"✔ {len(subjects)} subjects found")

# Check one subject
m = json.load(open(f"{PRE}/s01/meta.json"))
print(f"  s01 windows : {m['total_windows']}")
print(f"  train trials: {len(m['train_trials'])}")
print(f"  val trials  : {len(m['val_trials'])}")
print(f"  test trials : {len(m['test_trials'])}")

✔ 32 subjects found
  s01 windows : 1160
  train trials: 28
  val trials  : 6
  test trials : 6


In [8]:
import sys, types, torch, torch.nn as nn, torch.nn.functional as F
import time, gc, json
sys.path.insert(0, "/kaggle/working")

import pgcn_core as C
import pgcn_train as T
import pgcn_model as M
from pgcn_core import ResultsStore, DEAPDataset, make_scheduler, TASKS
from pgcn_model import build_model
from torch_geometric.loader import DataLoader as GeoLoader
import torch.optim as optim
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from pathlib import Path

# ── Simpler model ──
def build_small_model(mode="within", fusion="none", ablation="none"):
    abl = M.ABLATION_CONFIGS.get(ablation, M.ABLATION_CONFIGS["none"])
    return M.PyramidalGCN(
        in_dim=abl["in_dim"], hidden=64, heads=4, dropout=0.45,
        use_band_attn=abl["use_band_attn"],
        use_hierarchy=abl["use_hierarchy"],
        use_functional=abl["use_functional"],
        n_domains=0, fusion=fusion,
        edge_dropout=0.0, feat_dropout=0.0,
    )
M.build_model = build_small_model

# ── Simple cross-entropy multi-task loss ──
class CEMultiTaskLoss(nn.Module):
    def forward(self, out, y):
        if y.dim()==1: y=y.unsqueeze(0)
        if y.dim()==2 and y.size(1)!=5:
            y=y.view(out["val"].size(0),5)
        total=0.
        for i,t in enumerate(["val","aro","dom","lik"]):
            total = total + F.cross_entropy(out[t], y[:,i].long())
        total = total + 2.*F.cross_entropy(out["quad"], y[:,4].long())
        return total, {"total":total}
C.MultiTaskLoss = CEMultiTaskLoss
C.load_class_weights = lambda d,s: None

# ── Window-level evaluate ──
@torch.no_grad()
def eval_window(model, loader, device):
    model.eval()
    yt={t:[] for t in TASKS}; yp={t:[] for t in TASKS}
    for batch in loader:
        batch=batch.to(device,non_blocking=True)
        out=model(batch)
        y=batch.y
        if y.dim()==1: y=y.unsqueeze(0)
        yn=y.cpu().numpy()
        for i,t in enumerate(["val","aro","dom","lik"]):
            yt[t].append(yn[:,i])
            yp[t].append(out[t].argmax(1).cpu().numpy())
        yt["quad"].append(yn[:,4])
        yp["quad"].append(out["quad"].argmax(1).cpu().numpy())
    res={}
    for i,t in enumerate(["val","aro","dom","lik"]):
        a=np.concatenate(yt[t]); b=np.concatenate(yp[t])
        res[f"{t}_f1"]=f1_score(a,b,average="binary",zero_division=0)
        res[f"{t}_acc"]=accuracy_score(a,b)
    a=np.concatenate(yt["quad"]); b=np.concatenate(yp["quad"])
    res["quad_f1"]=f1_score(a,b,average="macro",zero_division=0)
    res["quad_acc"]=accuracy_score(a,b)
    return res

# ── Custom within-subject training ──
def run_within_fixed(args, store):
    device=torch.device(args.device)
    subjects=[f"s{i:02d}" for i in range(1,33)]
    subjects=[s for s in subjects
              if (Path(args.data_dir)/s/"meta.json").exists()]
    print(f"\n  Within-subject: {len(subjects)} subjects")

    cfg=dict(lr=3e-4, wd=1e-3, bs=32, ep=200, pat=30, wu=10, clip=1.)

    for subj in subjects:
        sdir=Path(args.out_dir)/subj
        ckpt=sdir/"best.pt"; rpath=sdir/"metrics.json"

        if rpath.exists() and not args.force:
            print(f"  [SKIP] {subj}")
            store.per_subject[subj]=json.load(open(rpath)); continue

        sdir.mkdir(parents=True,exist_ok=True)
        print(f"\n{'─'*55}\n  {subj}")

        ds_tr=DEAPDataset(args.data_dir,[subj],"train")
        ds_vl=DEAPDataset(args.data_dir,[subj],"val")
        ds_te=DEAPDataset(args.data_dir,[subj],"test")
        kw=dict(batch_size=cfg["bs"],num_workers=2,pin_memory=True)
        dl_tr=GeoLoader(ds_tr,shuffle=True,**kw)
        dl_vl=GeoLoader(ds_vl,shuffle=False,**kw)
        dl_te=GeoLoader(ds_te,shuffle=False,**kw)

        if len(ds_tr)==0: continue

        model=build_small_model("within","none",args.ablation).to(device)
        crit=CEMultiTaskLoss()
        opt=optim.AdamW(model.parameters(),lr=cfg["lr"],weight_decay=cfg["wd"])
        sched=make_scheduler(opt,cfg["wu"],cfg["ep"])
        scaler=torch.amp.GradScaler("cuda")

        best_f1=-1; wait=0

        for ep in range(1,cfg["ep"]+1):
            # Train
            model.train(); tot=0.; nb=0
            for batch in dl_tr:
                batch=batch.to(device,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda"):
                    out=model(batch); loss,_=crit(out,batch.y)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(),cfg["clip"])
                scaler.step(opt); scaler.update()
                tot+=loss.item(); nb+=1
            sched.step()
            tl=tot/max(1,nb)

            # Validate (window-level F1 for early stopping)
            vm=eval_window(model,dl_vl,device)
            # Use mean F1 across tasks as stopping criterion
            mean_f1=np.mean([vm["val_f1"],vm["aro_f1"],
                             vm["dom_f1"],vm["lik_f1"]])

            if ep%20==0 or ep==1:
                print(f"  ep{ep:3d}/{cfg['ep']}  tr={tl:.3f}  "
                      f"vl_val={vm['val_f1']:.3f}  vl_aro={vm['aro_f1']:.3f}  "
                      f"vl_quad={vm['quad_f1']:.3f}  mean={mean_f1:.3f}")

            # Early stop on F1 (higher is better)
            if mean_f1 > best_f1 + 1e-4:
                best_f1=mean_f1; wait=0
                torch.save(model.state_dict(),ckpt)
            else:
                wait+=1
                if wait>=cfg["pat"]:
                    print(f"  Early stop ep{ep}  best_mean_f1={best_f1:.4f}")
                    break

        # Test (window-level)
        model.load_state_dict(torch.load(ckpt,map_location=device,
                                          weights_only=True))
        tm=eval_window(model,dl_te,device)
        store.per_subject[subj]=tm
        json.dump(tm,open(rpath,"w"),indent=2)
        print(f"  TEST [window]  "
              f"val={tm['val_f1']:.4f}  aro={tm['aro_f1']:.4f}  "
              f"dom={tm['dom_f1']:.4f}  lik={tm['lik_f1']:.4f}  "
              f"quad={tm['quad_f1']:.4f}")

        del model,opt,sched,scaler
        torch.cuda.empty_cache(); gc.collect()

print("✔ All patches applied")

# ── Run ──
args = types.SimpleNamespace(
    data_dir = "/kaggle/input/datasets/jerinanan/pgcn-deap-preprocessed",
    out_dir  = "/kaggle/working/results_eeg",
    mode="within", fusion="none", ablation="none",
    device="cuda", workers=2, force=True,
)
store = ResultsStore()
run_within_fixed(args, store)
store.summary(out_dir=f"{args.out_dir}/within_none/aggregate")
print("\n✔ EEG-Only Within done!")

✔ All patches applied

  Within-subject: 32 subjects

───────────────────────────────────────────────────────
  s01
  DEAPDataset [train]     812 windows  (1 subjects)
  DEAPDataset [val  ]     174 windows  (1 subjects)
  DEAPDataset [test ]     174 windows  (1 subjects)
  ep  1/200  tr=6.218  vl_val=0.000  vl_aro=0.615  vl_quad=0.145  mean=0.591
  ep 20/200  tr=3.924  vl_val=0.341  vl_aro=0.508  vl_quad=0.228  mean=0.639
  ep 40/200  tr=2.828  vl_val=0.337  vl_aro=0.482  vl_quad=0.190  mean=0.612
  Early stop ep40  best_mean_f1=0.7743
  TEST [window]  val=0.5614  aro=0.7454  dom=0.9091  lik=1.0000  quad=0.2279

───────────────────────────────────────────────────────
  s02
  DEAPDataset [train]     812 windows  (1 subjects)
  DEAPDataset [val  ]     174 windows  (1 subjects)
  DEAPDataset [test ]     174 windows  (1 subjects)
  ep  1/200  tr=6.245  vl_val=0.749  vl_aro=0.800  vl_quad=0.272  mean=0.804
  ep 20/200  tr=4.783  vl_val=0.808  vl_aro=0.614  vl_quad=0.316  mean=0.789
  Early 

ValueError: need at least one array to concatenate

In [10]:
import os, json, numpy as np

# Correct path - subjects are directly in results_eeg
OUT = "/kaggle/working/results_eeg"

results = {}
for sdir in sorted(os.scandir(OUT), key=lambda x: x.name):
    rpath = f"{sdir.path}/metrics.json"
    if os.path.exists(rpath) and sdir.name.startswith("s"):
        results[sdir.name] = json.load(open(rpath))

print(f"✔ Completed: {len(results)}/32 subjects\n")
print("="*55)
print("  PER-SUBJECT MEAN ± STD  (EEG-Only Within-Subject)")
print("="*55)

for task, label in [("val_f1","Valence"), ("aro_f1","Arousal"),
                     ("dom_f1","Dominance"), ("lik_f1","Liking"),
                     ("quad_f1","Quadrant")]:
    vals = [v[task] for v in results.values() if task in v]
    print(f"  {label:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}  "
          f"(min={np.min(vals):.4f}  max={np.max(vals):.4f})")

print("\n  ACCURACY")
for task, label in [("val_acc","Valence"), ("aro_acc","Arousal"),
                     ("dom_acc","Dominance"), ("lik_acc","Liking"),
                     ("quad_acc","Quadrant")]:
    vals = [v[task] for v in results.values() if task in v]
    print(f"  {label:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

# Save
import pandas as pd
os.makedirs(f"{OUT}/aggregate", exist_ok=True)
df = pd.DataFrame(results).T
df.to_csv(f"{OUT}/aggregate/per_subject.csv", float_format="%.4f")
summary = {}
for task in ["val_f1","aro_f1","dom_f1","lik_f1","quad_f1",
             "val_acc","aro_acc","dom_acc","lik_acc","quad_acc"]:
    vals = [v[task] for v in results.values() if task in v]
    if vals:
        summary[f"{task}_mean"] = float(np.mean(vals))
        summary[f"{task}_std"]  = float(np.std(vals))
json.dump(summary, open(f"{OUT}/aggregate/global_metrics.json","w"), indent=2)
print(f"\n✔ Summary saved to {OUT}/aggregate/")

✔ Completed: 32/32 subjects

  PER-SUBJECT MEAN ± STD  (EEG-Only Within-Subject)
  Valence     : 0.6189 ± 0.1891  (min=0.1974  max=0.9091)
  Arousal     : 0.6669 ± 0.2051  (min=0.0385  max=0.9091)
  Dominance   : 0.6652 ± 0.2399  (min=0.1159  max=1.0000)
  Liking      : 0.7300 ± 0.1651  (min=0.4413  max=1.0000)
  Quadrant    : 0.2821 ± 0.1214  (min=0.1236  max=0.6890)

  ACCURACY
  Valence     : 0.5902 ± 0.1364
  Arousal     : 0.6309 ± 0.1302
  Dominance   : 0.6489 ± 0.1966
  Liking      : 0.6331 ± 0.1856
  Quadrant    : 0.4140 ± 0.1262

✔ Summary saved to /kaggle/working/results_eeg/aggregate/


In [11]:
import shutil, os

# Zip the results folder
shutil.make_archive(
    '/kaggle/working/results_eeg_within',  # output zip name
    'zip',
    '/kaggle/working',                      # root dir
    'results_eeg'                           # folder to zip
)
print(f"✔ Zipped: {os.path.getsize('/kaggle/working/results_eeg_within.zip')/1e6:.1f} MB")

✔ Zipped: 4.9 MB
